# Mock DMI, AIUPred, and Monte Carlo Tutorial

This notebook shows the three structural layers together on mocked
data:

1. predict domain-motif interactions,
2. optionally run AIUPred if the `idr` extra is installed,
3. run the standalone Monte Carlo motif filter on a deterministic
   residue-level score table.

The Monte Carlo section is guaranteed to run without network access
or external databases because it uses mocked residue scores.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists() and (path / 'microbiolink').exists():
            return path
    raise RuntimeError('Could not find the MicrobioLink repository root.')


repo_root = find_repo_root()
workdir = repo_root / 'tutorials' / 'mock_data' / 'runs' / '02_mock_dmi_aiupred_monte_carlo'
resources_dir = workdir / 'aiupred_resources'
results_dir = workdir / 'aiupred_results'

if workdir.exists():
    shutil.rmtree(workdir)

for path in [workdir, resources_dir, results_dir]:
    path.mkdir(parents = True, exist_ok = True)

repo_root, workdir

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(repo_root)],
    check = True,
    cwd = repo_root,
)

In [ ]:
import pandas as pd

from microbiolink.motif_monte_carlo_filter import run_monte_carlo_filter
from microbiolink_api import interactions_to_dataframe
from microbiolink_api import predict_domain_motif_interactions

## Step 1: Create mock inputs

Both mock host proteins begin with `MCR`, so both produce a real
packaged DMI hit. Later, only one of them will pass the Monte Carlo
structural filter.

In [ ]:
            human_fasta_file = workdir / 'mock_human_proteins.fasta'
            bacterial_domain_file = workdir / 'mock_bacterial_domains.tsv'

            human_fasta_file.write_text(
                '>sp|P12345|MOCK_POSITIVE
'
                'MCRAAAAAAAAAAAAAAAAAAAAAAAAAAA
'
                '>sp|Q99999|MOCK_NEGATIVE
'
                'MCRGGGGGGGGGGGGGGGGGGGGGGGGGGG
',
                encoding = 'utf-8',
            )

            bacterial_domain_file.write_text(
                'Entry	Pfam
'
                'BACPROT1	PF02207
',
                encoding = 'utf-8',
            )

            print(human_fasta_file)
            print(bacterial_domain_file)

## Step 2: Run the DMI step

In [ ]:
interactions = predict_domain_motif_interactions(
    fasta_file = human_fasta_file,
    bacterial_domain_file = bacterial_domain_file,
)

interaction_frame = interactions_to_dataframe(interactions)

dmi_table_file = workdir / 'mock_dmi_output.tsv'
dmi_legacy_file = workdir / 'mock_dmi_legacy.csv'

interaction_frame.to_csv(dmi_table_file, sep = '	', index = False)
interaction_frame.to_csv(dmi_legacy_file, sep = ';', index = False, header = False)

interaction_frame

## Step 3: Optionally run AIUPred

This section only runs if the notebook environment already has the
`idr` extra installed.

In [ ]:
aiupred_available = False
aiupred_error = None

try:
    from microbiolink import AIUPred
    aiupred_available = True
except Exception as error:
    aiupred_error = error

if aiupred_available:
    aiupred_output_name = 'mock_aiupred_output.tsv'
    AIUPred.main(
        [
            '--hmi_prediction',
            str(dmi_legacy_file),
            '--fasta_file',
            str(human_fasta_file),
            '--resources',
            str(resources_dir),
            '--results',
            str(results_dir),
            '--output',
            aiupred_output_name,
            '--force-cpu',
        ],
    )
    aiupred_result = results_dir / aiupred_output_name
else:
    aiupred_result = None

aiupred_available, repr(aiupred_error) if aiupred_error is not None else aiupred_result

## Step 4: Build a deterministic residue-level score table

The positive protein has one short high-confidence disordered and
binding-prone segment exactly where the motif sits. The negative
protein has the same DMI hit but no structural support.

In [ ]:
structure_scores_file = workdir / 'mock_structure_scores.tsv'

sequence_lengths = {
    'P12345': 30,
    'Q99999': 30,
}

structure_rows = []
for protein, length in sequence_lengths.items():
    for position in range(1, length + 1):
        if protein == 'P12345' and position <= 3:
            disorder_score = 0.95
            binding_score = 0.95
        else:
            disorder_score = 0.10
            binding_score = 0.10

        structure_rows.append(
            {
                'protein': protein,
                'position': position,
                'disorder_score': disorder_score,
                'binding_score': binding_score,
            },
        )

structure_frame = pd.DataFrame(structure_rows)
structure_frame.to_csv(structure_scores_file, sep = '	', index = False)
structure_frame.head()

## Step 5: Run the Monte Carlo filter

In [ ]:
monte_carlo_full_file = workdir / 'mock_monte_carlo_full.tsv'
monte_carlo_kept_file = workdir / 'mock_monte_carlo_kept.tsv'

monte_carlo_frame = run_monte_carlo_filter(
    interaction_file = dmi_table_file,
    output_file = monte_carlo_full_file,
    filtered_output_file = monte_carlo_kept_file,
    structure_scores = structure_scores_file,
    iterations = 5000,
    alpha = 0.05,
    disorder_threshold = 0.60,
    binding_threshold = 0.60,
    min_support_fraction = 1.0,
    require_binding = True,
    seed = 7,
)

monte_carlo_frame[
    [
        'human_protein',
        'motif',
        'start',
        'end',
        'observed_support_fraction',
        'monte_carlo_pvalue',
        'passes_monte_carlo',
    ]
]

In [ ]:
pd.read_csv(monte_carlo_kept_file, sep = '	')

## Result

In this mock setup, `P12345` passes the Monte Carlo filter and
`Q99999` does not. That gives you a minimal end-to-end example of:

- DMI generation,
- optional AIUPred execution,
- final structural motif filtering.